In [40]:
%matplotlib inline

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, FloatSlider, Output, Label, Dropdown, FloatLogSlider, HTML, Checkbox
from scipy.interpolate import RegularGridInterpolator
from IPython.display import display, clear_output, Audio
import sys
import PathGeneration as pg
from Interpolation import SplineTrajectory3D
import json
import cv2
from moviepy import VideoFileClip
import matplotlib as mpl

build_dir = "cppSimulation/build"
if build_dir not in sys.path:
    sys.path.insert(0, build_dir)


In [41]:
# audio utils

def db_scale(x, amin=1e-10):
    return 20.0 * np.log10(np.maximum(x, amin))

def lin_scale(x):
    return 10**(x / 20)

def get_audio_from_source(source, mono=True):
    if isinstance(source, str):
        data, sr = sf.read(source, always_2d=True)
        if data.size == 0:
            raise RuntimeError("Empty or unsupported audio file.")
        if mono:
            audio = data[:, 0]
        return (audio, sr)
    elif isinstance(source, tuple):
        return source
    else:
        raise TypeError("bad scene audio type")

In [42]:
# microphone arrays

class ArrayState:
    positions = np.array([[0, 0, 0]])
    cropac = {}
    calibration = np.array([])
    def __init__(self, pos=np.array([[0, 0, 0]]), cropac={}):
        self.positions = pos
        self.cropac = cropac

def load_array(path, calibrationPath):

    array = ArrayState()
    with open(path, "r", encoding="utf-8") as f:
        array.__dict__ = json.load(f)
    array.positions = np.stack((*np.array(array.positions).T, np.zeros(len(array.positions))))
    
    with open(calibrationPath, "r", encoding="utf-8") as f:
        array.calibration = np.array(json.load(f))

    array.positions[0] -= np.mean(array.positions[0])
    array.positions[1] -= np.mean(array.positions[1])
    
    return array

def default8_1():
    return load_array("default_array8_1.json", "default_array8_calibration.json")

def default8_2():
    return load_array("default_array8_2.json", "default_array8_calibration.json")

def circle8():
    return load_array("circle_array8.json", "circle_array8_calibration.json")


In [43]:
# beamforming utils

def polar_to_look_direction(aziv, zeniv):
    return np.stack((
        np.cos(aziv) * np.sin(zeniv),
        np.sin(aziv) * np.sin(zeniv),
        np.cos(zeniv)), axis=-1)

# half-sphere of look directions
def create_look_directions(n, fov):

    fov += 1e-5
    x = np.linspace(-fov, fov, n)
    y = np.linspace(-fov, fov, n)

    xv, yv = np.meshgrid(x, y)

    # circle the square
    # ratio = np.maximum(np.abs(xv), np.abs(yv)) / (np.sqrt(xv*xv + yv*yv) + 1e-18)
    # xv *= ratio
    # yv *= ratio
    
    xv = np.reshape(xv, -1)
    yv = np.reshape(yv, -1)
    aziv = np.atan2(yv, xv)
    zeniv = np.sqrt(xv*xv + yv*yv)

    unit_look_direction = polar_to_look_direction(aziv, zeniv)
    
    # xy = np.stack((xv, yv), axis = -1)

    return (unit_look_direction, aziv, zeniv, (x, y))

def to_polar_angles(point):
    x, y, z = point
    azi = np.atan2(y, x)
    zeni = np.atan2(np.sqrt(x*x+y*y), z)
    return (azi, zeni)

# look directions (dirs, 3), microphone positions (3, m). returns (dirs, m)
def create_distance_vectors(look_directions, microphone_positions):
    return look_directions @ microphone_positions

def create_steering_vectors(distance_vectors, k):
    return np.exp(1j * distance_vectors * k) / np.sqrt(distance_vectors.shape[-1])

# expects shape (samples, mics)
def resonator_filter(audio, halfTime, fc, sr):
    decay = np.pow(0.5, 1 / (halfTime * sr))
    n = audio.shape[0]
    m = audio.shape[1]
    a = np.full(m, np.exp(2j * np.pi * fc / sr) * decay)
    x = np.zeros(audio.shape,  dtype=np.complex64)
    for i in range(1, n):
        x[i] = x[i-1] * a + audio[i-1] * (1.0 - decay)
    return x

def analyze_audio(audio, t_idx, nfft, freq, sr):
    
    m, n = audio.shape
    x = np.zeros(m, dtype=np.complex64)
    R = np.zeros((m, m), dtype=np.complex64)

    if n < 2 * nfft:
        return (x, R)
    
    hann = (0.5 - 0.5 * np.cos(2 * np.pi / nfft * np.arange(nfft))) / nfft
    wavelet = np.exp(-2j * np.pi * freq / sr * np.arange(nfft)) * hann

    pos = max(2 * nfft, t_idx)

    K = 4 * m

    step = nfft // K
    for i in range(K * step, 0, -step):
        x = audio[:,pos-i-nfft:pos-i] @ wavelet
        R += np.conj(np.array([x]).T) @ np.array([x])
    
    R /= np.sqrt(np.abs(np.sum(np.conj(R)*R)))
    x /= np.sqrt(np.abs(np.sum(np.conj(x)*x)))

    return (x, R)

# def analyze_audio2(audio, t_idx, nfft, freq, sr):
    
    
#     m, n = audio.shape
#     x = np.zeros(m, dtype=np.complex64)
#     R = np.zeros((m, m), dtype=np.complex64)

#     if n < 2 * nfft:
#         return (x, R)
    
#     w1 = np.exp(-2j * np.pi * (freq / sr - 1 / nfft) * np.arange(2*nfft))
#     w2 = np.exp(-2j * np.pi * (freq / sr + 0) * np.arange(2*nfft))
#     w3 = np.exp(-2j * np.pi * (freq / sr + 1 / nfft) * np.arange(2*nfft))
    
#     pos = max(2 * nfft - 1, t_idx)
#     a = audio[:,pos-2*nfft+1:pos+1]
#     s1 = np.cumsum(np.hstack((np.zeros((m, 1)), w1 * a)), axis=-1)
#     s2 = np.cumsum(np.hstack((np.zeros((m, 1)), w2 * a)), axis=-1)
#     s3 = np.cumsum(np.hstack((np.zeros((m, 1)), w3 * a)), axis=-1)

#     step = nfft // (2 * m)
#     for i in range(2 * m * step, 0, -step):
#         x = (- 0.25 * (s1[:,2*nfft-i] - s1[:,nfft-1-i]) 
#              + 0.50 * (s2[:,2*nfft-i] - s2[:,nfft-1-i])
#              - 0.25 * (s3[:,2*nfft-i] - s3[:,nfft-1-i])) / nfft

#         R += np.array([x]).T @ np.array([x])
    
#     R /= 2 * m
#     return (x, R)

In [44]:
# Delay and sum

# x (m, w), steering_vectors (dirs, m)
def das_beamform(x, steering_vectors):
    return np.conj(steering_vectors) @ x

def das_beampattern(look_direction, steering_vectors):
    return das_beamform(look_direction, steering_vectors)


In [45]:
# MDF

def distance_steering_matrix(mic_array, k):
    n = mic_array.shape[1]
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            d = mic_array[:,i] - mic_array[:,j]
            dist[i,j] = np.sqrt(np.sum(d*d))
    return np.sinc(k * dist)

def mdf_weights(R, steering_vectors, dial=1.0):
    loading = dial * np.trace(R).real / R.shape[0]
    w = steering_vectors @ np.linalg.inv(R + loading * np.eye(R.shape[0]))
    norm = np.sum(np.conj(steering_vectors) * w, axis=-1)
    w /= norm[..., None]
    return w

def mdf_beamform(x, R, steering_vectors):
    w = mdf_weights(R, steering_vectors)
    return np.conj(w) @ x

def mdf_beampattern(look_direction, R, steering_vectors):
    w = mdf_weights(R, look_direction)
    return das_beamform(w, steering_vectors)

In [46]:
# MVDR

def mvdr_weights(R, steering_vectors, dial = 1.5):
    loading = dial * np.trace(R).real / R.shape[0]
    w = steering_vectors @ np.linalg.inv(R + loading * np.eye(R.shape[0]))
    norm = np.sum(np.conj(steering_vectors) * w, axis=-1)
    w /= norm[..., None]
    return w

def mvdr_beamform(x, R, steering_vectors):
    w = mvdr_weights(R, steering_vectors)
    return np.conj(w) @ x

def mvdr_beampattern(look_direction, R, steering_vectors):
    w = mvdr_weights(R, look_direction)
    return das_beamform(w, steering_vectors)

In [47]:
# CroPaC

def cropac_beamform(x, cropac_subsets, steering_vectors):
    z = das_beamform(x, steering_vectors)
    for (ix1, ix2) in cropac_subsets.values():
        z1 = das_beamform(x[ix1], steering_vectors[:,ix1])
        z2 = das_beamform(x[ix2], steering_vectors[:,ix2])
        gain = np.maximum(0, np.real(2 * z1 * np.conj(z2)) / np.real(z1 * np.conj(z1) + z2 * np.conj(z2) + 1e-18))
        z *= gain
    return z

def cropac_beampattern(look_direction, cropac_subsets, steering_vectors):
    return cropac_beamform(look_direction, cropac_subsets, steering_vectors)
    z = das_beamform(look_direction, steering_vectors)
    for (ix1, ix2) in cropac_subsets.values():
        z1 = das_beamform(look_direction[ix1], steering_vectors[:,ix1])
        z2 = das_beamform(look_direction[ix2], steering_vectors[:,ix2])
        gain = np.maximum(0, np.real(2 * z1 * np.conj(z2)) / np.real(z1 * np.conj(z1) + z2 * np.conj(z2) + 1e-18))
        z *= gain
    return z

In [48]:
# MUSIC

def music_beamform(R, steering_vectors):
    w = np.linalg.eigh(R)[1][:,-1]
    return np.conj(steering_vectors) @ np.conj(w)

def music_beampattern(R, steering_vectors):
    return music_beamform(R, steering_vectors)

In [54]:
# Low frequency CroPaC for an octagon microphone array

def planar_directions(directions):
    return np.atan2(directions[:,0], directions[:,1])

def low8_cropac_beamform(x, direction_vectors, mic_array):

    x = x / np.abs(x)

    mic = planar_directions(mic_array.T)
    dir = planar_directions(direction_vectors)
    relative_angles = dir[:, None] - mic[None, :]

    dipole_steering = np.cos(relative_angles) * 1j
    quadropole_steering = np.cos(2 * relative_angles)

    z1 = dipole_steering @ x
    z2 = quadropole_steering @ x

    # G = 2 * np.real(z1 * z2) / ()

    return np.maximum(0, np.real(z1 * np.conj(z2)))

def low8_cropac_beampattern(look_direction, steering_vectors, mic_array):

    mic = planar_directions(mic_array.T)
    dir = planar_directions(look_direction[None,:])
    relative_angles = dir[0] - mic

    dipole_steering = np.cos(relative_angles) * 1j
    quadropole_steering = np.cos(2 * relative_angles)

    z1 = steering_vectors @ dipole_steering
    z2 = steering_vectors @ quadropole_steering
    
    return np.maximum(0, np.real(z1 * np.conj(z2)))

In [55]:
# plot utils

def plot_3d_base(ax, azim, elev, size=7.5):
    """
    Plot a 3D flight path and a reference point at the origin.
    points: numpy array of shape (3, N)
    t: numpy array of shape (N,)
    """
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.view_init(elev=elev, azim=azim)

    ax.grid(True)

    ax.set_xlim(-size, size)
    ax.set_ylim(-size, size)
    ax.set_zlim(0, 2 * size)

def plot_path_3d(ax, points, t_idx, color):
    resolution = 2**12
    step = max(1, points.shape[1] // resolution)
    idx = np.arange(0, points.shape[1], step)
    if points.shape[1] % step != 0:
        idx = np.append(idx, [points.shape[1]-1]) 
    p = points[:,idx]
    x, y, z = p

    ax.plot(x, y, z, color=color)
    ax.plot(*points[:,t_idx], color=color, markersize=10, linestyle='', marker='o')

def plot_position_3d(ax, position, t_idx, color):
    if isinstance(position, tuple):
        plot_path_3d(ax, position[0], t_idx, color)
    else:
        ax.plot(*position, color=color, markersize=10, linestyle='', marker='o')

todeg = 1 / np.pi * 180

def plot_mesh_polar_on_image(ax, plot_points, z):

    aziv, zeniv = plot_points

    cm = mpl.colormaps['viridis']
    c = cm.colors
    c = [*[c[0] for i in range(len(c))], *c]
    c = [[*c[i], max(0, i/len(c)-0.4)] for i in range(len(c))]
    cm = mpl.colors.ListedColormap(c)

    # cm = mpl.colors.LinearSegmentedColormap.from_list("mycmap", [
    #         [0.8, 0.8, 0.1, 0.0],
    #         [0.8, 0.8, 0.1, 0.05],
    #         [0.8, 0.5, 0.1, 0.1],
    #         [1.0, 0.2, 0.2, 0.5]
    #     ])

    c = ax.pcolormesh(aziv, zeniv * todeg, np.abs(z), shading='gouraud', cmap=cm)
    ax.set_ylim([0, 90 + 1e-5])
    plt.colorbar(c, ax=ax)

def plot_mesh_polar(ax, plot_points, z):
    aziv, zeniv = plot_points
    c = ax.pcolormesh(aziv, zeniv * todeg, np.abs(z), shading='gouraud', cmap='viridis')
    ax.set_ylim([0, 90 + 1e-5])
    plt.colorbar(c, ax=ax)

In [66]:
# video utils

def video_duration_seconds(cap) -> float | None:

    fps = cap.get(cv2.CAP_PROP_FPS)
    nframes = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    # Try duration = frames / fps if both are valid
    if fps and fps > 0 and nframes and nframes > 0:
        duration = nframes / fps
    else:
        # Some backends expose duration directly (ms)
        duration = cap.get(cv2.CAP_PROP_DURATION) / 1000.0
        if not duration or duration <= 0:
            # Fallback: jump to last frame and read its timestamp
            if nframes and nframes > 0:
                cap.set(cv2.CAP_PROP_POS_FRAMES, nframes - 1)
                ok, _ = cap.read()
                if ok:
                    duration = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
                else:
                    duration = None
            else:
                duration = None

    return duration-1/fps

def moving_average(a, n=3):
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

def find_marker(audio: np.ndarray, sr):
    f = 5000
    marker = np.exp(np.arange(sr)*f/sr*2j*np.pi) / sr

    N = 1
    while N < audio.shape[0]+sr:
        N *= 2

    a = np.zeros(N, dtype=np.complex64)
    b = np.zeros(N, dtype=np.complex64)

    a[:audio.shape[0]] = audio[:,0]
    b[:sr] = marker

    y = np.abs(np.fft.ifft(np.fft.fft(a)*np.fft.fft(b)))
    y = moving_average(y, sr)

    max = np.max(y)
    for i in range(1, len(y)-1):
        if y[i] > y[i-1] and y[i] > y[i+1] and y[i] > max * 0.1:
            return i

    return np.argmax(y)

def audio_from_video_moviepy(video_path):
    with VideoFileClip(video_path) as clip:
        if clip.audio is None:
            return None, None
        arr = clip.audio.to_soundarray()  # float32, shape (n_samples, n_channels)
        sr = clip.audio.fps
        if arr.ndim == 1:
            arr = np.array([arr]).T
        return arr, sr

class Video:

    def __init__(self, video_path, array_audio, audio_sr):
        self.cap = cv2.VideoCapture(video_path)
        if not self.cap.isOpened():
            raise RuntimeError(f"Cannot open video: {video_path}")
        self.dur = video_duration_seconds(self.cap)

        video_audio, video_sr = audio_from_video_moviepy(video_path)

        self.t_sync = find_marker(array_audio, audio_sr) / audio_sr - find_marker(video_audio, video_sr) / video_sr

    def get_frame_at_time(self, t_sec: float) -> np.ndarray:
        """
        Return an RGB frame from video at time t_idx (seconds).
        Raises RuntimeError if frame cannot be read.
        """

        t_sec -= self.t_sync

        # Seek by milliseconds
        self.cap.set(cv2.CAP_PROP_POS_MSEC, max(0, min(t_sec, self.dur)) * 1000.0)
        ok, frame_bgr = self.cap.read()

        if not ok or frame_bgr is None:
            raise RuntimeError(f"Could not read frame at {t_sec} s")

        # Convert BGR (OpenCV) -> RGB (matplotlib)
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        return frame_rgb

def show_frame_as_background(ax, frame: np.ndarray):
    """
    Display frame as background image.
    - extent=None: axes coordinates will be pixel space [0..width] x [0..height] (origin at top-left).
    - extent=[xmin, xmax, ymin, ymax]: map frame to your data coordinates.
    Returns the axes used.
    """

    h, w = frame.shape[:2]

    # ax.imshow(frame, origin='upper')

    # Push image to the back

    d = 0.55
    im = ax.imshow(
        np.flip(np.transpose(frame[:h//2,:], [1, 0, 2]), [0, 1]),
        extent=[0.5-d , 0.5+d, 0.5-d, 0.5+d],       # fill the whole axes box
        transform=ax.transAxes,    # interpret extent in axes coords
        zorder=0,                  # behind other artists
        interpolation="bilinear",
    )

    # Clip the image to the circular patch of the polar axes
    im.set_clip_path(ax.patch)

    ax.set_zorder(0)
    return ax

In [68]:
c = 344
fig_width = 10
dpi = 80
beamforming_resolution = 100
plot_resolution = 100
fov = 90 / todeg
array_orientation = np.pi / 2

freqmin = 100
freqmax = 5000

audioPath = "mv4t3"
micArray = "circle8"

audioPaths = {
    "lowfreq": ("../samples/circle8_lowfreq.wav", "../videos/circle8_lowfreq.mov"),
    "ideal": ("../samples/idealdrone.wav", "../videos/idealdrone.mov"),
    "outwalk": ("../samples/outsidewalk2.wav", "../videos/outsidewalk2.mov"),
    "drone1": ("../samples/droneflight1.wav", "../videos/droneflight1.mov"),
    "drone2": ("../samples/droneflight2.wav", "../videos/droneflight2.mov"),
    "drone3": ("../samples/droneflight3.wav", "../videos/droneflight3.mov"),
    "drone4": ("../samples/droneflight4.wav", "../videos/droneflight4.mov"),
    "drone5": ("../samples/droneflight5.wav", "../videos/droneflight5.mov"),
    "drone6": ("../samples/droneflight6.wav", "../videos/droneflight6.mov"),
    "mv4t1": ("../samples/mavic4_test1.wav", "../videos/mavic4_test1.mov"),
    "mv4t2": ("../samples/mavic4_test2.wav", "../videos/mavic4_test2.mov"),
    "mv4t3": ("../samples/mavic4_test3.wav", "../videos/mavic4_test3.mov"),
    "testwalk": ("../samples/testwalk.wav", None),
}

microphone_arrays = {
    "circle8": circle8,
    "default8_1": default8_1,
    "default8_2": default8_2,
}

audio_path, video_path = audioPaths[audioPath]
mic_array = microphone_arrays[micArray]()
audio, sr = sf.read(audio_path, always_2d=True)
video = None
if video_path != None:
    video = Video(video_path, audio, sr)

total_samples =  audio.shape[0]
audio = audio.T
total_duration = total_samples / sr

look_directions, aziv, zeniv, xy = create_look_directions(beamforming_resolution, fov)
plot_points = (aziv.reshape((beamforming_resolution, beamforming_resolution)),
               zeniv.reshape((beamforming_resolution, beamforming_resolution)))
distance_vectors = create_distance_vectors(look_directions, mic_array.positions)

w_NFFT = Dropdown(options=[2**i for i in range(6, 14)], value=1024, description="NFFT:")
w_algo = Dropdown(options=["DaS", "MVDR", "MUSIC", "MDF", "CroPaC", "LCroPaC"], value="LCroPaC", description="Algorithm:")
for i in [w_NFFT, w_algo]:
    i.layout.width = f"{180}px"

w_time = FloatSlider(
    value=0.0, min=0.0, max=max(0.0, total_duration - 0.01),
    step=max(0.001, total_duration / 1000.0),
    description="Time (s)", continuous_update=True, readout_format=".2f"
)
w_freq = FloatSlider(
    value=1000, min=freqmin, max=freqmax, step=1,
    description="Freq (Hz)", continuous_update=True, readout_format=".0f"
)

w_mode = Dropdown(options=["Video", "Just Output", "Video Pickup", "Just Pickup"], value="Video", description="Mode:")

# w_azi = FloatSlider(
#     value=-60, min=-180, max=180, step=1,
#     description="Azi (deg)", continuous_update=True, readout_format=".0f"
# )
# w_ele = FloatSlider(
#     value=30, min=-90, max=90, step=1,
#     description="Ele (deg)", continuous_update=True, readout_format=".0f"
# )

for i in [w_time, w_freq]:
    i.layout.width = f"{fig_width * dpi * 1.2}px"

out1 = Output()
out2 = Output()

plt.rc('font', **{'size': 16})

def update_plot(change=None):
    time = max(0.0, min(w_time.value, total_duration))
    t_idx = int(round(time * sr))
    t_idx = max(0, min(t_idx, total_samples-1))

    mp = mic_array.positions
    dv = distance_vectors

    k = 2 * np.pi * w_freq.value / c
    steering_vectors = create_steering_vectors(dv, k)

    z_view = np.zeros(beamforming_resolution * beamforming_resolution)
    z_pickup = np.zeros(beamforming_resolution * beamforming_resolution)

    x, R = analyze_audio(audio, t_idx, int(w_NFFT.value), w_freq.value, sr)

    if w_algo.value == "DaS":
        z_view = das_beamform(x, steering_vectors)
    elif w_algo.value == "MVDR":
        z_view = mvdr_beamform(x, R, steering_vectors)
    elif w_algo.value == "MUSIC":
        z_view = music_beamform(R, steering_vectors)
    elif w_algo.value == "MDF":
        Rd = distance_steering_matrix(mp, k)
        z_view = mdf_beamform(x, Rd, steering_vectors)
    elif w_algo.value == "CroPaC":
        z_view = cropac_beamform(x, mic_array.cropac, steering_vectors)
    elif w_algo.value == "LCroPaC":
        z_view = low8_cropac_beamform(x, look_directions, mic_array.positions)

    z_view = z_view.reshape((beamforming_resolution, beamforming_resolution))

    target_idx = np.argmax(np.abs(z_view))
    target_azi = aziv[target_idx]
    target_zeni = zeniv[target_idx]
    unit_target_direction = polar_to_look_direction(target_azi, target_zeni)
    v0 = create_steering_vectors(create_distance_vectors(np.array([unit_target_direction]), mp), k)
    target_zeni = min(target_zeni * todeg, 90)

    if w_algo.value == "DaS":
        z_pickup = das_beampattern(v0[0], steering_vectors)
    elif w_algo.value == "MVDR":
        z_pickup = mvdr_beampattern(v0[0], R, steering_vectors)
    elif w_algo.value == "MUSIC":
        z_pickup = music_beampattern(R, steering_vectors)
    elif w_algo.value == "MDF":
        Rd = distance_steering_matrix(mp, k)
        z_pickup = mdf_beampattern(v0[0], Rd, steering_vectors)
    elif w_algo.value == "CroPaC":
        z_pickup = cropac_beampattern(v0[0], mic_array.cropac, steering_vectors)
    elif w_algo.value == "LCroPaC":
        z_pickup = low8_cropac_beampattern(unit_target_direction, steering_vectors, mic_array.positions)

    z_pickup = z_pickup.reshape((beamforming_resolution, beamforming_resolution))

    N = int(w_NFFT.value)
    s_idx = max(0, min(t_idx - N, total_samples - N))
    e_idx = s_idx + N
    hann = (0.5 - 0.5 * np.cos(2 * np.pi / N * np.arange(N))) / N
    mini_x = audio[0,s_idx:e_idx] * hann
    mini_f = np.fft.rfftfreq(N, 1/sr)
    mini_X = np.abs(np.fft.rfft(mini_x * np.hanning(N))) / N

    for i in range(N):
        mini_X[i] = 0.0
        if mini_f[i] > freqmin:
            break

    mini_max = np.max(mini_X)

    with out1:
        fig = plt.figure(figsize=(fig_width, fig_width*0.1))
        ax_top = fig.add_subplot(111)
        ax_top.fill_between(mini_f, np.zeros(N//2+1), mini_X, color="0.6", alpha=0.6, linewidth=0)
        ax_top.plot([0, freqmax], [0, 0], color="0.2", linewidth=0.5)  # zero line
        ax_top.axvline(w_freq.value, color="C3", linestyle="--", linewidth=1.5)
        ax_top.set_xlim(0, freqmax)
        ax_top.set_ylim(-0.05 * mini_max, 1.05 * mini_max)
        ax_top.set_axis_off()

    with out2:
        clear_output(wait=True)
        fig = plt.figure(figsize=(fig_width, fig_width*0.7))

        ax_view = fig.add_subplot(111, projection='polar')

        if w_mode.value in ["Video", "Video Pickup"] and video != None:
            show_frame_as_background(ax_view, video.get_frame_at_time(time))
            if w_mode.value == "Video Pickup":
                plot_mesh_polar_on_image(ax_view, plot_points, z_pickup)
            else:
                plot_mesh_polar_on_image(ax_view, plot_points, z_view)
        else:
            if w_mode.value == "Just Pickup":
                plot_mesh_polar(ax_view, plot_points, z_pickup)
            else:
                plot_mesh_polar(ax_view, plot_points, z_view)
        # ax_view.set_ylim(90)
        ax_view.scatter(target_azi, target_zeni, color='red', s=200, marker='x')
        # ax_view.set_title("Acoustic view")

        # ax_pickup = fig.add_subplot(122, projection='polar')
        # plot_mesh_polar(ax_pickup, plot_points, z_pickup)
        # ax_pickup.scatter(target_azi, target_zeni, color='red', s=100, marker='x')
        # ax_pickup.set_title("Pickup pattern")
        
        plt.tight_layout()
        plt.show()

        l_idx = max(t_idx - sr//2, 0)
        r_idx = min(l_idx + sr, total_samples)
        l_idx = r_idx - sr
        display(Audio(audio[0, l_idx:r_idx], rate=sr))

for w in (w_time, w_freq, w_NFFT, w_algo, w_mode):
    w.observe(update_plot, names="value")


display(VBox([
    HBox([w_algo, w_NFFT, w_mode]),
    w_time,
    w_freq,
    out1,
    out2
]))

update_plot()